# 01 · MCP 연결 · 인증 · 툴 목록 확인

**파이프라인 1/4** — Teams / Mail MCP 서버에 로그인하고 연결을 확인합니다.

이 노트북은 **pipeline 패키지를 import하지 않고**, 아래 셋업 셀 하나에 설정·인증·MCP 연결
로직을 인라인해 **노트북만으로 전체 흐름을 이해·실행**할 수 있게 했습니다.

하는 일:
1. `.env`(테넌트/클라이언트 ID/AUTH_MODE/LLM)를 읽습니다.
2. 두 소스(Mail/Teams)에 **device-code 로그인**을 1회 수행합니다 → 토큰은
   `.token_cache.json`에 저장되어 이후 노트북과 웹앱이 조용히 재사용합니다.
3. 각 MCP 서버의 **툴 목록**을 가져와 연결을 검증합니다.

> **사전 준비**: 프로젝트 루트의 `.env.example`을 `.env`로 복사하고 `TENANT_ID`, `CLIENT_ID`,
> LLM 설정을 채우세요. device-code 로그인을 위해 Entra 앱에 **"Allow public client flows" = Yes**,
> Agent Tools 위임 권한 **관리자 동의**가 필요합니다. 자세한 절차는 **`../SETUP.md`**.

## 셋업 (설정 + 인증)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')
TOKEN_CACHE = PROJECT_ROOT / '.token_cache.json'   # 로그인 1회 → 이후 노트북/웹앱이 재사용

# 2) 설정값 (.env에서 읽음)
TENANT_ID     = os.environ.get('TENANT_ID') or '00000000-0000-0000-0000-000000000000'
CLIENT_ID     = os.environ.get('CLIENT_ID', '')
CLIENT_SECRET = os.environ.get('CLIENT_SECRET', '')
AUTH_MODE     = os.environ.get('AUTH_MODE', 'device_code')   # device_code | client_credentials
AUTHORITY     = f'https://login.microsoftonline.com/{TENANT_ID}'

def _mcp_url(env_var, server_id):
    return (os.environ.get(env_var) or '').strip() or \
        f'https://agent365.svc.cloud.microsoft/agents/tenants/{TENANT_ID}/servers/{server_id}'

# 두 MCP 소스: Mail / Teams. 서버마다 delegated 스코프가 달라 토큰도 소스별로 1개씩.
SOURCES = {
    'mail':  {'label': 'Mail',  'url': _mcp_url('MAIL_MCP_SERVER_URL',  'mcp_MailTools')},
    'teams': {'label': 'Teams', 'url': _mcp_url('TEAMS_MCP_SERVER_URL', 'mcp_TeamsServer')},
}
for _s in SOURCES.values():
    _s['scopes'] = [f"{_s['url']}/.default"]

# 3) MSAL 인증 — 소스별 access token. device_code는 최초 1회만 로그인, 이후 캐시 재사용.
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token(source_key: str) -> str:
    """소스 1개의 access token 확보 (없으면 device-code 로그인)."""
    assert CLIENT_ID, 'CLIENT_ID가 .env에 없습니다 (.env.example 참고).'
    scopes = SOURCES[source_key]['scopes']
    if AUTH_MODE == 'client_credentials':           # 앱 전용 토큰
        app = msal.ConfidentialClientApplication(
            CLIENT_ID, authority=AUTHORITY, client_credential=CLIENT_SECRET, token_cache=_cache)
        res = app.acquire_token_for_client(scopes=scopes)
    else:                                           # 위임(사용자) 로그인
        app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
        accounts = app.get_accounts()
        res = app.acquire_token_silent(scopes, account=accounts[0]) if accounts else None
        if not (res and 'access_token' in res):     # 캐시에 없으면 device-code 로그인
            flow = app.initiate_device_flow(scopes=scopes)
            print(flow['message'])                   # https://microsoft.com/devicelogin + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TENANT_ID   :', TENANT_ID, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
for _k, _s in SOURCES.items():
    print(f"  [{_k}] {_s['label']} -> {_s['url']}")

In [ ]:
# --- MCP 읽기 헬퍼: 연결은 호출할 때마다 열고 항상 닫는다 ---
async def _with_session(source_key, token, fn):
    url = SOURCES[source_key]['url']
    # 최신 MCP SDK: 인증 헤더는 httpx 클라이언트에 담아 streamable_http_client에 전달
    async with create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}) as http_client:
        async with streamable_http_client(url, http_client=http_client) as (r, w, _):
            async with ClientSession(r, w) as s:      # MCP 세션 초기화(handshake)
                await s.initialize()
                return await fn(s)

async def list_tools(token, source_key):
    """MCP 서버가 노출하는 도구(함수) 목록."""
    res = await _with_session(source_key, token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, source_key, name, args=None):
    """MCP 도구를 이름+인자로 직접 호출."""
    return await _with_session(source_key, token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def content_to_text(result) -> str:
    """MCP 도구 결과의 content 블록들을 읽기 좋은 문자열로 평탄화."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            parts.append(getattr(b, 'text', ''))
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

## 1. 인증 방법 선택 & Entra 앱 설정

로그인 방식은 두 가지입니다. **기본은 방법 1(device-code)** 이며, 바로 아래 셀에서 한 줄로 바꿀 수 있습니다.
(영구 설정은 `.env`의 `AUTH_MODE`로 하면 `02`~`04` 노트북에도 함께 적용됩니다.)

### 방법 1 — device-code (위임 로그인, 권장·기본)
사용자 계정으로 1회 로그인합니다. Work IQ Mail/Teams MCP가 **위임 스코프**를 사용하므로 대부분의 테넌트에서 이 방식이 맞습니다.

**필수 사전설정 — 앱 등록에 "공개 클라이언트 흐름 허용"을 켜야 합니다.**
안 켜면 로그인 자체는 성공해도 토큰 교환에서 다음 오류가 납니다:
`AADSTS7000218: The request body must contain the following parameter: 'client_assertion' or 'client_secret'.`

**포털**
1. **Entra/Azure Portal → App registrations**(앱 등록) — *Enterprise applications 아님!*
2. `.env`의 `CLIENT_ID`와 일치하는 앱 선택 (안 보이면 상단 **All applications**에서 CLIENT_ID로 검색)
3. **Manage → Authentication**
4. 맨 아래 **Advanced settings → "Allow public client flows" → Yes → Save**

> ⚠️ 이 옵션은 **App registrations**(애플리케이션 개체)에만 있습니다. **Enterprise applications**
> (서비스 주체)에는 Authentication 메뉴가 없어 보이지 않습니다.

**CLI** (권한 있는 계정)
```bash
az ad app update --id <CLIENT_ID> --is-fallback-public-client true
az ad app show   --id <CLIENT_ID> --query isFallbackPublicClient -o tsv   # → true
```

### 방법 2 — client_credentials (앱 전용 토큰)
로그인 창 없이 앱이 직접 토큰을 받습니다. **`CLIENT_SECRET`** 과 MCP 리소스에 대한
**애플리케이션 권한(관리자 동의)** 이 필요합니다. 위임 스코프만 허용된 테넌트에서는 대부분 실패합니다.

- 앱 등록 → **Certificates & secrets** 에서 시크릿 생성 → `.env`의 `CLIENT_SECRET`에 저장
- 앱 등록 → **API permissions** 에서 해당 리소스의 *Application* 권한 추가 + 관리자 동의

> 자세한 절차·권한 ID 조회·문제 해결은 **[`../SETUP.md`](../SETUP.md)** 참고.

In [ ]:
# 인증 방법 선택 (기본 = 방법 1: device-code)
#   'device_code'        → 방법 1 (권장). Entra 앱에 "공개 클라이언트 흐름 허용 = 예" 필요
#   'client_credentials' → 방법 2. CLIENT_SECRET + 애플리케이션 권한(관리자 동의) 필요
AUTH_MODE = 'device_code'          # 방법 2로 바꾸려면 'client_credentials'

# 방법 2일 때만 사용. 비워두면 .env의 CLIENT_SECRET을 그대로 씀 (주석 해제 시 직접 지정)
# CLIENT_SECRET = 'your-client-secret'

assert AUTH_MODE in ('device_code', 'client_credentials'), \
    "AUTH_MODE는 'device_code' 또는 'client_credentials'"
if AUTH_MODE == 'client_credentials' and not CLIENT_SECRET:
    print('⚠ 방법 2(client_credentials) 선택됨 — CLIENT_SECRET이 비어 있습니다.')
    print('  .env의 CLIENT_SECRET을 채우거나 위 주석을 해제하세요.')
print('선택된 인증 방법:', AUTH_MODE,
      '(방법 1: device-code)' if AUTH_MODE == 'device_code' else '(방법 2: client_credentials)')

## 2. 설정 확인
`.env`가 제대로 읽혔는지, 두 MCP 소스 URL이 맞는지 확인합니다.

In [ ]:
print('LLM 설정 :', 'Azure' if os.environ.get('AZURE_OPENAI_DEPLOYMENT') else
      ('OpenAI' if os.environ.get('OPENAI_API_KEY') else '없음(02/04 에이전트 실행 불가)'))
print('토큰캐시 :', TOKEN_CACHE, '(존재:', TOKEN_CACHE.exists(), ')')
assert CLIENT_ID, 'CLIENT_ID가 비어 있습니다 — .env를 확인하세요.'
print('설정 OK')

## 3. 로그인 (선택한 방법으로)
위에서 고른 `AUTH_MODE`로 소스별 토큰을 확보합니다.
- **방법 1(device_code)**: 소스별 로그인 안내(`https://microsoft.com/devicelogin` + 코드)가 출력됩니다. 브라우저에서 코드를 입력해 로그인하면 토큰이 캐시에 저장되고, 이미 캐시가 있으면 즉시 통과합니다.
- **방법 2(client_credentials)**: 창 없이 앱 전용 토큰을 바로 받습니다.

In [ ]:
tokens = {}
for key in SOURCES:
    print(f'\n=== 로그인: {key} ({SOURCES[key]["label"]}) ===')
    try:
        tokens[key] = ensure_token(key)
        print(f'  OK — 토큰 확보 (len={len(tokens[key])})')
    except Exception as exc:
        print(f'  실패 — {exc}')
print('\n확보된 소스:', list(tokens.keys()))

## 4. 연결 검증 — 툴 목록 조회
각 MCP 서버에 연결해 사용 가능한 툴(함수) 목록을 가져옵니다. 여기서 확인한 툴 이름을
02(샘플)·03(직접 호출)에서 활용합니다.

In [ ]:
tools_by_source = {}
for key, token in tokens.items():
    try:
        ts = await list_tools(token, key)
        tools_by_source[key] = ts
        print(f'\n=== {key}: {len(ts)}개 툴 ===')
        for t in ts:
            desc = (getattr(t, 'description', '') or '').splitlines()[0][:80]
            print(f'  - {t.name}: {desc}')
    except Exception as exc:
        print(f'{key} 툴 목록 조회 실패: {exc}')

특정 툴의 입력 스키마(파라미터)를 보고 싶다면:

In [ ]:
for key, ts in tools_by_source.items():
    print(f'\n########## {key} ##########')
    for t in ts[:3]:
        print(f'\n### {t.name}')
        print(json.dumps(tool_schema(t), ensure_ascii=False, indent=2)[:1200])

✅ 여기까지 성공했다면 인증·연결 완료입니다. `.token_cache.json`이 생성되어 이후
노트북과 웹앱(`app/main.py`)이 이 토큰을 재사용합니다. → **02_seed_sample_data**로 이동.